# FPL 2025/26 Monte Carlo Season Simulation

Runs N=30 strategy configurations × 3 random seeds against actual 2025/26 GW points.
Uses walk-forward LightGBM predictions (retrained every 3 GWs) for decisions only;
scoring uses real Vaastav data throughout.

**Do not modify existing `bot/` files.** This notebook is self-contained.

In [ ]:
import subprocess, sys
pkgs = [
    'git+https://github.com/DH4410/fpl-auto.git',
    'lightgbm>=4.3',
    'scikit-learn>=1.4',
    'matplotlib>=3.8',
    'seaborn>=0.13',
    'tqdm>=4.66',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('All packages installed')


In [ ]:
from __future__ import annotations
import logging, warnings
from typing import Optional

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

from bot.data_collector import DataCollector
import bot.feature_engineering as fe
from bot.fpl_rules import (
    FPLRules, GKP, DEF, MID, FWD,
    SQUAD_SIZE, SQUAD_COMPOSITION, STARTING_BUDGET_TENTHS,
    MAX_PLAYERS_PER_CLUB, FORMATION_MINIMUMS, FORMATION_MAXIMUMS,
    HIT_COST, MAX_BANKED_FT, FREE_TRANSFERS_PER_GW,
    CAPTAIN_MULTIPLIER, TRIPLE_CAPTAIN_MULTIPLIER,
    CHIP_WILDCARD, CHIP_FREE_HIT, CHIP_TRIPLE_CAPTAIN, CHIP_BENCH_BOOST,
    FIRST_HALF_GWS, SECOND_HALF_GWS,
)

RULES = FPLRules()
_POS_MAP = {'GKP': GKP, 'DEF': DEF, 'MID': MID, 'FWD': FWD}
SIM_SEASON = '2025-26'
print('Imports OK')


In [ ]:
dc = DataCollector()
history = dc.load_multi_season_history()

h = history[history['season'] == SIM_SEASON]
gw_stats = h['GW'].agg(['min', 'max', 'nunique'])
print(f'2025-26: {len(h):,} rows | GW {gw_stats["min"]}–{gw_stats["max"]} '
      f'({gw_stats["nunique"]} unique GWs) | {h["element"].nunique()} players')

assert int(gw_stats['nunique']) == 38, f'Expected 38 GWs, got {gw_stats["nunique"]}'

# Sanity: top scorers GW1
name_col = next((c for c in ('name', 'web_name', 'second_name') if c in h.columns), None)
cols = ['element'] + ([name_col] if name_col else []) + ['total_points', 'minutes']
top5 = h[h['GW'] == 1].nlargest(5, 'total_points')[cols]
print(f'\nTop GW1 scorers:\n{top5.to_string(index=False)}')
print('\n✓ 38 GWs confirmed')


In [ ]:
print('Computing EWMA features (may take ~30s)...')
feat_df = fe.compute_ewma_stats(history, alpha=0.25)
print('Computing rolling-window features...')
feat_df = fe.rolling_window_stats(feat_df)

# Leakage check: first GW of every player must have NaN EWMA
ewma_cols = [c for c in feat_df.columns if c.startswith('ewma_')]
first_round = feat_df.groupby('element')['round'].min().rename('first_round')
check = feat_df.join(first_round, on='element')
first_rows = check[check['round'] == check['first_round']]
nan_frac = first_rows[ewma_cols].isna().all(axis=1).mean()
assert nan_frac == 1.0, f'Leakage: {1-nan_frac:.1%} of first-GW rows have non-NaN EWMA'
print(f'✓ Leakage check passed ({len(ewma_cols)} EWMA cols NaN on first appearance)')

feat_cols = [c for c in feat_df.columns if c.startswith(('ewma_', 'roll3_', 'roll6_'))]
feat_df[feat_cols] = feat_df[feat_cols].fillna(0)  # NaN → 0 for model input
print(f'Feature columns: {len(feat_cols)}')


In [ ]:
TARGET = 'total_points'
TRAIN_SEASONS = [s for s in feat_df['season'].unique() if s != SIM_SEASON]
STEP = 3  # refit every STEP GWs

gws_sim = sorted(feat_df.loc[feat_df['season'] == SIM_SEASON, 'GW'].unique())

def train_lgb(X, y):
    m = lgb.LGBMRegressor(
        n_estimators=400, learning_rate=0.04, num_leaves=63,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    m.fit(X, y, callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    return m

predictions: dict[tuple[int,int], float] = {}
spearman_log = []

print(f'Walk-forward: GW{gws_sim[0]}–GW{gws_sim[-1]}, refitting every {STEP} GWs')
print(f'Training on {len(TRAIN_SEASONS)} prior seasons + 25/26 lag')

for gw_start in range(gws_sim[0], gws_sim[-1] + 1, STEP):
    gw_block = [g for g in gws_sim if gw_start <= g < gw_start + STEP]

    train_mask = (
        (feat_df['season'].isin(TRAIN_SEASONS)) |
        ((feat_df['season'] == SIM_SEASON) & (feat_df['GW'] < gw_start))
    )
    tr = feat_df[train_mask].dropna(subset=[TARGET])
    model = train_lgb(tr[feat_cols].values, tr[TARGET].values)

    for gw in gw_block:
        gdf = feat_df[(feat_df['season'] == SIM_SEASON) & (feat_df['GW'] == gw)]
        if gdf.empty:
            continue
        preds = np.maximum(0.0, model.predict(gdf[feat_cols].values))
        r, _ = spearmanr(preds, gdf[TARGET].values)
        spearman_log.append({'gw': gw, 'spearman': r})
        print(f'  GW{gw:2d}: Spearman={r:.3f} (trained on {len(tr):,} rows)')
        for elem, pred in zip(gdf['element'].values, preds):
            predictions[(int(elem), int(gw))] = float(pred)

mean_r = float(np.mean([s['spearman'] for s in spearman_log]))
print(f'\nMean Spearman r = {mean_r:.3f} (target 0.60–0.70; higher = possible leakage)')
pred_df = pd.DataFrame(
    [{'element': e, 'gw': g, 'xpts': p} for (e, g), p in predictions.items()]
)
print(f'Prediction table: {len(pred_df):,} rows')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chip timing schedules: preferred GWs (first match wins)
# ─────────────────────────────────────────────────────────────────────────────
CHIP_PREFS: dict[str, dict[str, dict[str, list[int]]]] = {
    'early': {
        'h1': {CHIP_WILDCARD: [3,4,5], CHIP_FREE_HIT: [7,8,9],
               CHIP_TRIPLE_CAPTAIN: [9,10,11], CHIP_BENCH_BOOST: [12,13,14]},
        'h2': {CHIP_WILDCARD: [22,23,24], CHIP_FREE_HIT: [25,26,27],
               CHIP_TRIPLE_CAPTAIN: [28,29,30], CHIP_BENCH_BOOST: [31,32,33]},
    },
    'mid': {
        'h1': {CHIP_WILDCARD: [7,8,9], CHIP_FREE_HIT: [10,11,12],
               CHIP_TRIPLE_CAPTAIN: [13,14,15], CHIP_BENCH_BOOST: [16,17,18]},
        'h2': {CHIP_WILDCARD: [25,26,27], CHIP_FREE_HIT: [28,29,30],
               CHIP_TRIPLE_CAPTAIN: [31,32,33], CHIP_BENCH_BOOST: [34,35,36]},
    },
    'late': {
        'h1': {CHIP_WILDCARD: [14,15,16], CHIP_FREE_HIT: [17,18,19],
               CHIP_TRIPLE_CAPTAIN: [17,18,19], CHIP_BENCH_BOOST: [19]},
        'h2': {CHIP_WILDCARD: [31,32,33], CHIP_FREE_HIT: [34,35,36],
               CHIP_TRIPLE_CAPTAIN: [36,37,38], CHIP_BENCH_BOOST: [37,38]},
    },
    'optimal': {
        'h1': {CHIP_WILDCARD: [8,9,10], CHIP_FREE_HIT: [16,17,18],
               CHIP_TRIPLE_CAPTAIN: [11,12,13], CHIP_BENCH_BOOST: [14,15,16]},
        'h2': {CHIP_WILDCARD: [27,28,29], CHIP_FREE_HIT: [33,34,35],
               CHIP_TRIPLE_CAPTAIN: [36,37,38], CHIP_BENCH_BOOST: [30,31,32]},
    },
}


class MonteCarloSimulator:
    """Full-season FPL simulator for Monte Carlo strategy analysis.

    Decisions (squad picks, captains, transfers) use walk-forward ML predictions.
    Scoring uses actual Vaastav 2025/26 points.
    No live FPL API calls; no modifications to existing bot/ code.
    """

    JITTER = 0.05  # xpts noise for tie-breaking

    def __init__(self, history: pd.DataFrame, pred_df: pd.DataFrame,
                 season: str = '2025-26',
                 budget_tenths: int = STARTING_BUDGET_TENTHS):
        self.season = season
        self.budget_tenths = budget_tenths

        s = history[history['season'] == season].copy()

        # Normalise element_type to int 1-4
        if 'element_type' in s.columns:
            s['element_type'] = s['element_type'].apply(
                lambda x: _POS_MAP.get(str(x), int(x)) if not str(x).isdigit() else int(x)
            ).astype(int)
        elif 'position' in s.columns:
            s['element_type'] = s['position'].map(_POS_MAP).fillna(MID).astype(int)
        else:
            raise ValueError('Need element_type or position column')

        # Normalise team to int
        if s['team'].dtype == object:
            teams = sorted(s['team'].dropna().unique())
            tmap = {t: i + 1 for i, t in enumerate(teams)}
            s['team_id'] = s['team'].map(tmap).fillna(0).astype(int)
        else:
            s['team_id'] = s['team'].astype(int)

        s['value'] = pd.to_numeric(s['value'], errors='coerce').fillna(50).astype(int)
        s['element'] = s['element'].astype(int)
        s = s[s['element_type'].isin([GKP, DEF, MID, FWD])]
        self.sdf = s

        self.gws = sorted(s['GW'].unique())

        # Build fast lookup tables
        self._pts: dict[tuple[int,int], int] = {}
        self._mins: dict[tuple[int,int], int] = {}
        for r in s.itertuples(index=False):
            k = (int(r.element), int(r.GW))
            self._pts[k] = int(r.total_points)
            self._mins[k] = int(r.minutes)

        # Price at START of GW = previous GW's value (no look-ahead)
        self._gw_price: dict[tuple[int,int], int] = {}
        for elem, grp in s.groupby('element'):
            g = grp.sort_values('GW')
            vs, gws_e = g['value'].values, g['GW'].values
            for i, gw in enumerate(gws_e):
                self._gw_price[(int(elem), int(gw))] = int(vs[max(0, i - 1)])

        # Prediction lookup: (element, gw) → xpts
        self._pred: dict[tuple[int,int], float] = {
            (int(r.element), int(r.gw)): float(r.xpts)
            for r in pred_df.itertuples(index=False)
        }

    # ── pool helpers ─────────────────────────────────────────────────────────

    def _pool(self, gw: int, rng: np.random.RandomState) -> pd.DataFrame:
        """GW player pool with predictions and prices."""
        p = self.sdf[self.sdf['GW'] == gw].copy()
        p['xpts'] = p['element'].apply(lambda e: self._pred.get((int(e), int(gw)), 0.0))
        p['xpts_j'] = p['xpts'] + rng.uniform(-self.JITTER, self.JITTER, len(p))
        p['cur_price'] = p['element'].apply(
            lambda e: self._gw_price.get((int(e), int(gw)),
                      int(p.loc[p['element'] == e, 'value'].iat[0]) if (p['element'] == e).any() else 45)
        )
        return p.reset_index(drop=True)

    def _cur_price(self, pool: pd.DataFrame) -> dict[int, int]:
        return {int(r.element): int(r.cur_price) for r in pool.itertuples(index=False)}

    def _xpts(self, pool: pd.DataFrame) -> dict[int, float]:
        return {int(r.element): float(r.xpts_j) for r in pool.itertuples(index=False)}

    # ── squad building ───────────────────────────────────────────────────────

    def _build_squad(self, pool: pd.DataFrame, budget: int, style: str) -> list[dict]:
        """Greedy squad selection within budget and FPL constraints."""
        p = pool.copy()
        if style == 'premium':
            p['_key'] = p['xpts_j'] * np.where(p['cur_price'] >= 100, 1.15, 0.85)
        elif style == 'budget':
            p['_key'] = p['xpts_j'] / (p['cur_price'] / 10.0 + 0.1)
        else:
            p['_key'] = p['xpts_j']
        p = p.sort_values('_key', ascending=False)

        squad, pos_cnt, team_cnt, spent = [], {GKP:0,DEF:0,MID:0,FWD:0}, {}, 0

        for _, row in p.iterrows():
            pos, tid, price = int(row['element_type']), int(row['team_id']), int(row['cur_price'])
            if pos_cnt.get(pos, 0) >= SQUAD_COMPOSITION.get(pos, 0): continue
            if team_cnt.get(tid, 0) >= MAX_PLAYERS_PER_CLUB: continue
            if spent + price > budget: continue
            squad.append({'element': int(row['element']), 'element_type': pos, 'team': tid})
            pos_cnt[pos] = pos_cnt.get(pos, 0) + 1
            team_cnt[tid] = team_cnt.get(tid, 0) + 1
            spent += price
            if len(squad) == SQUAD_SIZE:
                break

        # Fallback: fill positions still short with cheapest valid players
        if len(squad) < SQUAD_SIZE:
            elems_in = {s['element'] for s in squad}
            for pos, need in SQUAD_COMPOSITION.items():
                while pos_cnt.get(pos, 0) < need:
                    cands = p[(p['element_type'] == pos) & (~p['element'].isin(elems_in))].sort_values('cur_price')
                    filled = False
                    for _, row in cands.iterrows():
                        tid, price = int(row['team_id']), int(row['cur_price'])
                        if team_cnt.get(tid, 0) >= MAX_PLAYERS_PER_CLUB: continue
                        if spent + price > budget: continue
                        squad.append({'element': int(row['element']), 'element_type': pos, 'team': tid})
                        elems_in.add(int(row['element']))
                        pos_cnt[pos] = pos_cnt.get(pos, 0) + 1
                        team_cnt[tid] = team_cnt.get(tid, 0) + 1
                        spent += price
                        filled = True
                        break
                    if not filled:
                        break  # cannot fill this position

        return squad

    # ── XI selection ─────────────────────────────────────────────────────────

    def _pick_xi(self, squad: list[dict], xpts: dict[int,float],
                 chip: Optional[str] = None) -> tuple[list[dict], list[dict]]:
        """Return (starting_xi, bench) as lists of player dicts."""
        if chip == CHIP_BENCH_BOOST:
            return list(squad), []  # score_gameweek handles BB correctly

        gkps = sorted([p for p in squad if p['element_type'] == GKP],
                      key=lambda p: xpts.get(p['element'], 0.0), reverse=True)
        out = sorted([p for p in squad if p['element_type'] != GKP],
                     key=lambda p: xpts.get(p['element'], 0.0), reverse=True)

        xi_out, pos_cnt, remaining = [], {DEF:0,MID:0,FWD:0}, list(out)
        for pos in [DEF, MID, FWD]:
            for p in list(remaining):
                if p['element_type'] == pos and pos_cnt[pos] < FORMATION_MINIMUMS[pos]:
                    xi_out.append(p); pos_cnt[pos] += 1; remaining.remove(p)
        for p in list(remaining):
            if len(xi_out) >= 10: break
            pos = p['element_type']
            if pos_cnt.get(pos, 0) < FORMATION_MAXIMUMS.get(pos, 5):
                xi_out.append(p); pos_cnt[pos] = pos_cnt.get(pos, 0) + 1; remaining.remove(p)

        xi = [gkps[0]] + xi_out
        bench_out = sorted(remaining, key=lambda p: xpts.get(p['element'], 0.0), reverse=True)
        bench = bench_out + ([gkps[1]] if len(gkps) > 1 else [])
        return xi, bench

    def _pick_captain(self, xi: list[dict], xpts: dict[int,float], strategy: str) -> tuple[int,int]:
        """Return (captain_id, vice_id)."""
        ranked = sorted(xi, key=lambda p: xpts.get(p['element'], 0.0), reverse=True)
        if strategy == 'differential':
            # 3rd-highest xpts as differential captain
            cap = ranked[2]['element'] if len(ranked) > 2 else ranked[0]['element']
            vc = ranked[0]['element']
        elif strategy == 'top_premium':
            # 2nd-highest xpts (contrarian; often a premium)
            cap = ranked[1]['element'] if len(ranked) > 1 else ranked[0]['element']
            vc = ranked[0]['element']
        else:  # top_xpts (default)
            cap = ranked[0]['element']
            vc = ranked[1]['element'] if len(ranked) > 1 else cap
        return cap, vc

    # ── transfer logic ───────────────────────────────────────────────────────

    def _best_swap(self, squad: list[dict], pool: pd.DataFrame,
                   buy_prices: dict[int,int], bank: int, xpts: dict[int,float],
                   prices: dict[int,int]) -> Optional[tuple[dict, object, float]]:
        """Find best single transfer. Returns (out_player, in_row, gain) or None."""
        squad_elems = {p['element'] for p in squad}
        team_cnt = {}
        for p in squad: team_cnt[p['team']] = team_cnt.get(p['team'], 0) + 1

        best = None
        for pos in [GKP, DEF, MID, FWD]:
            pos_squad = [p for p in squad if p['element_type'] == pos]
            if not pos_squad: continue
            out = min(pos_squad, key=lambda p: xpts.get(p['element'], 0.0))
            out_x = xpts.get(out['element'], 0.0)

            buy_t = buy_prices.get(out['element'], prices.get(out['element'], 45))
            cur_t = prices.get(out['element'], buy_t)
            sell_t = FPLRules.calc_selling_price_tenths(buy_t, cur_t)
            avail = bank + sell_t

            # Adjusted team counts after removing out player
            adj_team = dict(team_cnt)
            adj_team[out['team']] = adj_team.get(out['team'], 1) - 1

            cands = pool[
                (pool['element_type'] == pos) &
                (~pool['element'].isin(squad_elems - {out['element']})) &
                (pool['cur_price'] <= avail) &
                (pool['element'] != out['element'])
            ].copy()
            cands = cands[cands['team_id'].apply(lambda t: adj_team.get(t, 0) < MAX_PLAYERS_PER_CLUB)]

            if cands.empty: continue
            in_row = cands.nlargest(1, 'xpts_j').iloc[0]
            gain = float(in_row['xpts_j']) - out_x
            if best is None or gain > best[2]:
                best = (out, in_row, gain)

        return best

    def _execute_swap(self, squad: list[dict], out_p: dict, in_row,
                      buy_prices: dict[int,int], bank: int,
                      prices: dict[int,int]) -> tuple[list[dict], int]:
        """Execute one transfer. Mutates buy_prices in place."""
        buy_t = buy_prices.get(out_p['element'], prices.get(out_p['element'], 45))
        cur_t = prices.get(out_p['element'], buy_t)
        sell_t = FPLRules.calc_selling_price_tenths(buy_t, cur_t)
        new_price = int(in_row['cur_price'])
        new_bank = bank + sell_t - new_price
        new_squad = [p for p in squad if p['element'] != out_p['element']]
        new_elem = int(in_row['element'])
        new_squad.append({'element': new_elem, 'element_type': int(in_row['element_type']),
                          'team': int(in_row['team_id'])})
        buy_prices[new_elem] = new_price
        return new_squad, new_bank

    # ── chip scheduling ──────────────────────────────────────────────────────

    def _decide_chip(self, gw: int, chips_used: dict[str,bool], timing: str) -> Optional[str]:
        h = 'h1' if gw in FIRST_HALF_GWS else 'h2'
        sched = CHIP_PREFS[timing][h]
        # Priority order: TC and BB first (single-GW impact), then FH, WC
        for chip in [CHIP_TRIPLE_CAPTAIN, CHIP_BENCH_BOOST, CHIP_FREE_HIT, CHIP_WILDCARD]:
            if chips_used.get(f'{chip}_{h}', False): continue
            if gw in sched.get(chip, []): return chip
        return None

    # ── main simulation loop ─────────────────────────────────────────────────

    def run(self, strategy: dict, seed: int = 0) -> list[dict]:
        """Simulate one full season with the given strategy. Returns per-GW dicts."""
        rng = np.random.RandomState(seed)
        cap_strat   = strategy['captain_strategy']
        trans_strat = strategy['transfer_strategy']
        chip_timing = strategy['chip_timing']
        style       = strategy['squad_style']

        chips_used: dict[str,bool] = {}
        # Free-hit state: save/restore squad
        fh_restore: Optional[tuple[list[dict], dict[int,int], int]] = None

        # Initial squad (GW1)
        pool1 = self._pool(self.gws[0], rng)
        squad = self._build_squad(pool1, self.budget_tenths, style)
        buy_prices: dict[int,int] = {
            p['element']: self._gw_price.get((p['element'], self.gws[0]), 45)
            for p in squad
        }
        bank = self.budget_tenths - sum(buy_prices.values())
        ft = FREE_TRANSFERS_PER_GW  # free transfers available THIS gw
        cumul = 0
        results = []

        for gw in self.gws:
            pool = self._pool(gw, rng)
            prices = self._cur_price(pool)
            xpts   = self._xpts(pool)

            # Restore squad after Free Hit
            if fh_restore is not None:
                squad, buy_prices, bank = fh_restore
                fh_restore = None

            chip = self._decide_chip(gw, chips_used, chip_timing)
            n_transfers = 0

            if chip == CHIP_WILDCARD:
                # Rebuild squad; unlimited free transfers
                wc_budget = min(self.budget_tenths, bank + sum(
                    FPLRules.calc_selling_price_tenths(
                        buy_prices.get(p['element'], prices.get(p['element'], 45)),
                        prices.get(p['element'], buy_prices.get(p['element'], 45)),
                    ) for p in squad
                ))
                old_elems = {p['element'] for p in squad}
                squad = self._build_squad(pool, wc_budget, style)
                new_elems = {p['element'] for p in squad}
                n_transfers = len(new_elems - old_elems)
                buy_prices = {p['element']: prices.get(p['element'], 45) for p in squad}
                bank = wc_budget - sum(buy_prices.values())

            elif chip == CHIP_FREE_HIT:
                fh_budget = min(self.budget_tenths, bank + sum(
                    FPLRules.calc_selling_price_tenths(
                        buy_prices.get(p['element'], prices.get(p['element'], 45)),
                        prices.get(p['element'], buy_prices.get(p['element'], 45)),
                    ) for p in squad
                ))
                old_elems = {p['element'] for p in squad}
                fh_squad = self._build_squad(pool, fh_budget, style)
                n_transfers = len({p['element'] for p in fh_squad} - old_elems)
                fh_restore = (squad, buy_prices, bank)  # save for next GW
                squad = fh_squad

            else:
                # Regular transfer
                if trans_strat != 'passive':
                    swap = self._best_swap(squad, pool, buy_prices, bank, xpts, prices)
                    if swap is not None:
                        out_p, in_row, gain = swap
                        threshold = {'hit_if_gain_gt8': 8.0, 'hit_if_gain_gt12': 12.0}.get(trans_strat, 0.0)
                        # Take the transfer if: (a) have FT and gain > 0, or (b) gain > hit threshold
                        worth_hit = (trans_strat != '1ft_only') and (gain > threshold)
                        if gain > 0 and (ft >= 1 or worth_hit):
                            squad, bank = self._execute_swap(squad, out_p, in_row, buy_prices, bank, prices)
                            n_transfers = 1

            # Pick XI and captain
            xi, bench = self._pick_xi(squad, xpts, chip)
            cap_id, vc_id = self._pick_captain(xi, xpts, cap_strat)

            # Actual GW points
            pts_by = {p['element']: self._pts.get((p['element'], gw), 0) for p in squad}
            min_by = {p['element']: self._mins.get((p['element'], gw), 0) for p in squad}

            res = RULES.score_gameweek(
                starting_xi=xi, bench=bench,
                points_by_player=pts_by,
                captain_id=cap_id, vice_id=vc_id,
                played_minutes=min_by,
                chip=chip, n_transfers=n_transfers, free_transfers=ft,
            )

            cumul += res['total']
            ft = RULES.roll_free_transfers(ft, n_transfers)
            if chip:
                h_key = 'h1' if gw in FIRST_HALF_GWS else 'h2'
                chips_used[f'{chip}_{h_key}'] = True

            cap_actual = pts_by.get(cap_id, 0)
            vc_actual  = pts_by.get(vc_id, 0)

            results.append({
                'gw': gw,
                'pts': res['total'],
                'xi_pts': res['xi_points'],
                'cap_id': cap_id, 'cap_pts': cap_actual, 'cap_bonus': res['captain_points'],
                'vc_id': vc_id, 'vc_pts': vc_actual,
                'bench_pts': res['bench_points'],
                'hits': res['hits'],
                'n_transfers': n_transfers,
                'chip': chip or '',
                'ft_next': ft,
                'bank': bank / 10.0,
                'cumulative': cumul,
                'cap_beat_vc': cap_actual >= vc_actual,
            })

        return results

    def run_all(self, strategies: list[dict], n_seeds: int = 3) -> pd.DataFrame:
        """Run all configs × seeds. Returns combined DataFrame."""
        rows = []
        for i, strat in enumerate(tqdm(strategies, desc='Strategies')):
            label = (f"{strat['captain_strategy']}|"
                     f"{strat['transfer_strategy']}|"
                     f"{strat['chip_timing']}|"
                     f"{strat['squad_style']}")
            for seed in range(n_seeds):
                sim_id = f'sim{i:02d}_s{seed}'
                try:
                    gw_rows = self.run(strat, seed=seed)
                    for r in gw_rows:
                        r.update({
                            'sim_id': sim_id, 'strategy': label, 'seed': seed,
                            'captain_strategy': strat['captain_strategy'],
                            'transfer_strategy': strat['transfer_strategy'],
                            'chip_timing': strat['chip_timing'],
                            'squad_style': strat['squad_style'],
                        })
                    rows.extend(gw_rows)
                except Exception as exc:
                    print(f'  ERROR {sim_id}: {exc}')
        return pd.DataFrame(rows)


print('MonteCarloSimulator defined')


In [ ]:
from itertools import product as iproduct

# Grid of strategy parameters
CAPTAIN_STRATS   = ['top_xpts', 'differential', 'top_premium']
TRANSFER_STRATS  = ['1ft_only', 'hit_if_gain_gt8', 'passive', 'hit_if_gain_gt12']
CHIP_TIMINGS     = ['optimal', 'early', 'mid', 'late']
SQUAD_STYLES     = ['balanced', 'premium', 'budget']

# Build 30 distinct configs (subset of full grid)
STRATEGIES: list[dict] = []
for cap, tran, chip, style in iproduct(CAPTAIN_STRATS[:2], TRANSFER_STRATS[:2],
                                        CHIP_TIMINGS[:2], SQUAD_STYLES[:2]):
    STRATEGIES.append({'captain_strategy': cap, 'transfer_strategy': tran,
                        'chip_timing': chip, 'squad_style': style})

# Add variety: all transfer strategies × remaining captain strategies
for tran in TRANSFER_STRATS[2:]:
    for cap in CAPTAIN_STRATS:
        for chip in CHIP_TIMINGS[:2]:
            STRATEGIES.append({'captain_strategy': cap, 'transfer_strategy': tran,
                                'chip_timing': chip, 'squad_style': 'balanced'})

# Trim to exactly 30
STRATEGIES = STRATEGIES[:30]
print(f'{len(STRATEGIES)} strategy configs defined')
pd.DataFrame(STRATEGIES)


In [ ]:
N_SEEDS = 3
print(f'Running {len(STRATEGIES)} configs × {N_SEEDS} seeds = {len(STRATEGIES)*N_SEEDS} simulations...')

sim = MonteCarloSimulator(history, pred_df, season=SIM_SEASON)
results_df = sim.run_all(STRATEGIES, n_seeds=N_SEEDS)

print(f'\nDone. {len(results_df):,} GW rows | {results_df["sim_id"].nunique()} simulations')
results_df.head(5)


In [ ]:
# ── Per-simulation summary ────────────────────────────────────────────────────
sim_sum = (
    results_df
    .groupby(['sim_id','strategy','captain_strategy','transfer_strategy',
               'chip_timing','squad_style','seed'])
    .agg(
        total_pts   =('pts',       'sum'),
        total_hits  =('hits',      'sum'),
        bench_pts   =('bench_pts', 'sum'),
        cap_bonus   =('cap_bonus', 'sum'),
        pct_cap_won =('cap_beat_vc','mean'),
        chips_played=('chip',      lambda x: (x != '').sum()),
    ).reset_index()
)

# ── Config-level stats (mean ± std across seeds) ──────────────────────────────
cfg = (
    sim_sum
    .groupby(['captain_strategy','transfer_strategy','chip_timing','squad_style'])
    .agg(
        mean_pts =('total_pts','mean'),
        std_pts  =('total_pts','std'),
        min_pts  =('total_pts','min'),
        max_pts  =('total_pts','max'),
        mean_hits=('total_hits','mean'),
        mean_bench=('bench_pts','mean'),
    ).reset_index().sort_values('mean_pts', ascending=False)
)

print('=== Strategy Leaderboard ===')
print(cfg.to_string(index=False))

print('\n=== Top 5 ===')
print(cfg.head(5)[['captain_strategy','transfer_strategy','chip_timing',
                    'squad_style','mean_pts','std_pts','mean_hits']].to_string(index=False))

print('\n=== Bottom 5 ===')
print(cfg.tail(5)[['captain_strategy','transfer_strategy','chip_timing',
                    'squad_style','mean_pts','std_pts','mean_hits']].to_string(index=False))

# ── Chip attribution ──────────────────────────────────────────────────────────
chip_rows = results_df[results_df['chip'] != '']
if not chip_rows.empty:
    chip_attr = chip_rows.groupby('chip').agg(
        n=('pts','count'), mean_gw_pts=('pts','mean'), mean_bench=('bench_pts','mean')
    ).reset_index()
    print('\n=== Chip Attribution (mean GW points in the week the chip was played) ===')
    print(chip_attr.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
palette = sns.color_palette('muted')

# 1. Total-points histogram
ax = axes[0, 0]
sim_sum['total_pts'].hist(bins=20, ax=ax, color=palette[0], alpha=0.8, edgecolor='white')
mu = sim_sum['total_pts'].mean()
ax.axvline(mu, color='red', linestyle='--', label=f'Mean {mu:.0f}')
ax.set_title('Total Points Distribution'); ax.set_xlabel('Points'); ax.legend()

# 2. By captain strategy
ax = axes[0, 1]
g = sim_sum.groupby('captain_strategy')['total_pts'].agg(['mean','std'])
g['mean'].plot(kind='bar', ax=ax, yerr=g['std'], capsize=5, color=palette[1], alpha=0.8)
ax.set_title('Mean Pts by Captain Strategy'); ax.tick_params(axis='x', rotation=30)

# 3. By transfer strategy
ax = axes[0, 2]
g = sim_sum.groupby('transfer_strategy')['total_pts'].agg(['mean','std'])
g['mean'].plot(kind='bar', ax=ax, yerr=g['std'], capsize=5, color=palette[2], alpha=0.8)
ax.set_title('Mean Pts by Transfer Strategy'); ax.tick_params(axis='x', rotation=30)

# 4. By chip timing
ax = axes[1, 0]
g = sim_sum.groupby('chip_timing')['total_pts'].agg(['mean','std'])
g['mean'].plot(kind='bar', ax=ax, yerr=g['std'], capsize=5, color=palette[3], alpha=0.8)
ax.set_title('Mean Pts by Chip Timing'); ax.tick_params(axis='x', rotation=30)

# 5. By squad style
ax = axes[1, 1]
g = sim_sum.groupby('squad_style')['total_pts'].agg(['mean','std'])
g['mean'].plot(kind='bar', ax=ax, yerr=g['std'], capsize=5, color=palette[4], alpha=0.8)
ax.set_title('Mean Pts by Squad Style'); ax.tick_params(axis='x', rotation=30)

# 6. GW-average pts across all sims
ax = axes[1, 2]
gw_mean = results_df.groupby('gw')['pts'].agg(['mean','std'])
ax.plot(gw_mean.index, gw_mean['mean'], color=palette[5])
ax.fill_between(gw_mean.index, gw_mean['mean']-gw_mean['std'],
                gw_mean['mean']+gw_mean['std'], alpha=0.3, color=palette[5])
ax.set_title('Mean GW Points (all sims)'); ax.set_xlabel('GW'); ax.set_ylabel('Points')

plt.suptitle('FPL 2025/26 Monte Carlo — Strategy Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('mc_results.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved mc_results.png')


In [ ]:
# ── Top 3 simulations: week-by-week breakdown ─────────────────────────────────
top3_ids = sim_sum.nlargest(3, 'total_pts')['sim_id'].tolist()

for rank, sid in enumerate(top3_ids, 1):
    info = sim_sum[sim_sum['sim_id'] == sid].iloc[0]
    print(f"\n{'='*72}")
    print(f'#{rank}  {sid}  |  {info["strategy"]}')
    print(f'Total: {info["total_pts"]:.0f} pts  |  '
          f'Hits: {info["total_hits"]}  |  '
          f'Bench: {info["bench_pts"]:.0f}  |  '
          f'Cap bonus: {info["cap_bonus"]:.0f}')
    print(f'{"="*72}')

    gw_data = results_df[results_df['sim_id'] == sid].sort_values('gw')
    hdr = f'{"GW":>3}  {"Pts":>5}  {"Cumul":>6}  {"CapID":>7}  {"CapPts":>6}  {"×":>1}  {"HitCost":>7}  {"FT":>3}  {"Chip":<10}'
    print(hdr)
    print('-' * 72)

    for _, row in gw_data.iterrows():
        hit_str = f'-{int(row["hits"])}' if row['hits'] > 0 else ''
        chip_str = row['chip'] or '-'
        cap_mult = 3 if row['chip'] == CHIP_TRIPLE_CAPTAIN else 2
        print(f"{int(row['gw']):>3}  "
              f"{int(row['pts']):>5}  "
              f"{int(row['cumulative']):>6}  "
              f"{int(row['cap_id']):>7}  "
              f"{int(row['cap_pts']):>6}  "
              f"{cap_mult:>1}x "
              f"{hit_str:>7}  "
              f"{int(row['ft_next']):>3}  "
              f"{chip_str:<10}")

# ── Final leaderboard ─────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print('FINAL LEADERBOARD — all simulations ranked by total points')
print(f'{"="*72}')
lb = sim_sum.sort_values('total_pts', ascending=False)[
    ['sim_id','captain_strategy','transfer_strategy','chip_timing',
     'squad_style','seed','total_pts','total_hits','pct_cap_won']].reset_index(drop=True)
lb.index += 1
print(lb.to_string())


In [ ]:
# ── Step 5: Key insights ──────────────────────────────────────────────────────
best_cfg = cfg.iloc[0]
worst_cfg = cfg.iloc[-1]

print('=== INSIGHTS ===')
print(f'\nBest config:  {best_cfg["captain_strategy"]} | {best_cfg["transfer_strategy"]} | '
      f'{best_cfg["chip_timing"]} | {best_cfg["squad_style"]}')
print(f'  Mean pts: {best_cfg["mean_pts"]:.0f} ± {best_cfg["std_pts"]:.0f}')

print(f'\nWorst config: {worst_cfg["captain_strategy"]} | {worst_cfg["transfer_strategy"]} | '
      f'{worst_cfg["chip_timing"]} | {worst_cfg["squad_style"]}')
print(f'  Mean pts: {worst_cfg["mean_pts"]:.0f} ± {worst_cfg["std_pts"]:.0f}')

gap = best_cfg['mean_pts'] - worst_cfg['mean_pts']
print(f'\nGap best vs worst: {gap:.0f} pts ({gap/38:.1f} pts/GW)')

# Which dimension matters most?
for dim in ['captain_strategy','transfer_strategy','chip_timing','squad_style']:
    g = sim_sum.groupby(dim)['total_pts'].mean()
    rng = g.max() - g.min()
    best_val = g.idxmax()
    print(f'  {dim:22s}: range {rng:.0f} pts (best={best_val})')

# Consistency: low std = skill, high std = luck
print(f'\nLuck vs skill (std across seeds per config):')
print(f'  Mean std: {cfg["std_pts"].mean():.0f} pts')
print(f'  Max std:  {cfg["std_pts"].max():.0f} pts (highest luck sensitivity)')
print(f'  Min std:  {cfg["std_pts"].min():.0f} pts (most consistent)')
print(f'  → Configs with std < {cfg["std_pts"].quantile(0.25):.0f} are robustly good')

robust = cfg[cfg['std_pts'] <= cfg['std_pts'].quantile(0.25)].head(5)
print('\nRobust top strategies (high mean, low variance):')
print(robust[['captain_strategy','transfer_strategy','chip_timing','squad_style',
              'mean_pts','std_pts']].to_string(index=False))
